# LANGFENS — RQ1: P-SUP B-Control Official Training on Kaggle Tesla T4x2

Private execution notebook under CC BY-NC 4.0.
Executes strictly on authorized remote GPU hardware (Tesla T4) under CUDA FP16 AMP.
Strict fallback prohibition: Never fall back to local CPU/GPU training.
Capped at exactly 3,600 successful optimizer updates with per-epoch atomic remote mirroring.

## Experimental Arm: Matched Binary Control B-CONTROL
- Objective: `binary_collapsed` (collapsed binary logit: logsumexp(sub,del) - correct) + realized CTC loss
- Capacity: Exactly 266,667 parameters in `CompactOperationHead` matching treatment
- Partitions:
  - FIT (1,799 utterances across 12 speakers): Training gradient updates
  - SELECT (300 utterances across 2 speakers: MBMPS, TLV): Checkpoint selection & validation AP
  - CAL (300 utterances across 2 speakers: NCC, THV): Calibration
  - VAL (300 utterances across 2 speakers: NJS, RRBI): Preserved strictly unseen for evaluation
- Remote Target: `minkhoaa/langfens-rq1-cross-fold-checkpoints` under prefix `psup/b_control/`


In [ ]:
# Phase 1: Hardware & Environment Verification
import subprocess, sys

subprocess.run(["nvidia-smi"], check=True)

import torch
print(f"[GPU] CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("FAIL-CLOSED: CUDA GPU is not available on this runtime!")
print(f"[GPU] Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"[GPU {i}]: {torch.cuda.get_device_name(i)}")
assert "Tesla T4" in torch.cuda.get_device_name(0), (
    f"FAIL-CLOSED: Expected Tesla T4 accelerator, got {torch.cuda.get_device_name(0)}"
)

import yaml, transformers, huggingface_hub
print(f"[ENV] PyTorch version: {torch.__version__}")
print(f"[ENV] Transformers version: {transformers.__version__}")
print(f"[ENV] Hugging Face Hub version: {huggingface_hub.__version__}")


In [ ]:
# Phase 2: Credentials & Secrets Resolution (Strict Kaggle Secrets Path)
import os

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token and token.strip():
        os.environ["HF_TOKEN"] = token.strip()
        print("[AUTH] Successfully loaded HF_TOKEN from Kaggle Secrets.")
except Exception as e:
    print(f"[AUTH NOTE] Kaggle Secrets UserSecretsClient query: {e}")

if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"].strip():
    raise RuntimeError(
        "FAIL-CLOSED: HF_TOKEN is required in Kaggle Secrets (or environment) for private remote mirroring. "
        "Attach HF_TOKEN under Add-ons -> Secrets in the Kaggle notebook."
    )
print("[AUTH] Verified HF_TOKEN presence in environment for private remote mirroring.")


In [ ]:
# Phase 3: Clone, Setup PYTHONPATH, and Execute Training
import hashlib, os, subprocess, sys
from pathlib import Path

# Execution Gate: Set to True for immediate training execution upon Save & Run All
EXECUTE_TRAINING = True

if not EXECUTE_TRAINING:
    print("=" * 80)
    print("[HOLD] Execution gate EXECUTE_TRAINING is False.")
    print("[HOLD] Training execution is held pending explicit operator authorization.")
    print("=" * 80)
    sys.exit(0)

# 1. Clone/stage repository and checkout training branch
repo_dir = Path("/kaggle/working/repo")
if not repo_dir.exists():
    print("[GIT] Cloning repository into /kaggle/working/repo...")
    subprocess.run(
        ["git", "clone", "https://github.com/minkhoaa/Langfens-RQ1-Speech-Research.git", str(repo_dir)],
        check=True
    )

os.chdir(str(repo_dir))
subprocess.run(["git", "fetch", "--all"], cwd=repo_dir, check=True)

PINNED_COMMIT_SHA = "dd2dd8953fc5d946c0b88d72e915be53e7cd0d9f"
print(f"[GIT] Checking out pinned training commit: {PINNED_COMMIT_SHA}...")
subprocess.run(["git", "checkout", PINNED_COMMIT_SHA], cwd=repo_dir, check=True)

actual_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip()
print(f"[GIT] Verified Training Commit HEAD: {actual_head}")
assert actual_head == PINNED_COMMIT_SHA, f"FAIL-CLOSED: Checked out commit {actual_head} != expected {PINNED_COMMIT_SHA}"

# 2. Setup PYTHONPATH
sys.path.insert(0, str(repo_dir / "src"))
sys.path.insert(0, str(repo_dir))
os.environ["PYTHONPATH"] = f"{repo_dir}/src:{repo_dir}"

# 3. Robust Fail-Closed Dataset Discovery & Cryptographic Verification
EXPECTED_ARCHIVE_SIZE = 761922918
EXPECTED_ARCHIVE_SHA256 = "b750eb3bf3971afccd9ab9058a64fa6b3f5781399b9ef0f1294d29bf580784a9"

input_root = Path("/kaggle/input")
candidate_dirs = [
    Path("/kaggle/input/datasets/minkhoaa/langfens-rq1-fold1-staging"),
    Path("/kaggle/input/langfens-rq1-fold1-staging"),
]
if input_root.exists():
    candidate_dirs.extend(list(input_root.glob("**/langfens-rq1-fold1-staging")))
    candidate_dirs.extend(list(input_root.glob("*fold1-staging*")))
    candidate_dirs.extend(list(input_root.glob("*langfens-rq1-fold1*")))

staging_dir = None
for cand in candidate_dirs:
    if cand.is_dir():
        staging_dir = cand
        break

if staging_dir is None:
    available = list(input_root.rglob("*")) if input_root.exists() else []
    raise AssertionError(
        f"FAIL-CLOSED: Dataset input minkhoaa/langfens-rq1-fold1-staging not mounted!\n"
        f"Checked candidates: {[str(c) for c in candidate_dirs]}\n"
        f"Available under /kaggle/input: {[str(p) for p in available[:50]]}"
    )

print(f"[DATASET] Located staging directory: {staging_dir}")
print(f"[DATASET] Staging contents: {[p.name for p in staging_dir.iterdir()]}")

# Locate the authorized archive inside the staging directory:
# look for authorized_RQ1-TRN-02A_fold1_train_val_only.bin OR authorized_RQ1-TRN-02A_fold1_train_val_only.zip
# (or any *.bin / *.zip matching size 761,922,918 bytes)
archive_candidate = None
for fname in ("authorized_RQ1-TRN-02A_fold1_train_val_only.bin", "authorized_RQ1-TRN-02A_fold1_train_val_only.zip"):
    p = staging_dir / fname
    if p.is_file():
        archive_candidate = p
        break

if archive_candidate is None:
    for p in list(staging_dir.glob("*.bin")) + list(staging_dir.glob("*.zip")):
        if p.is_file() and p.stat().st_size == EXPECTED_ARCHIVE_SIZE:
            archive_candidate = p
            break

if archive_candidate is None:
    for p in list(staging_dir.rglob("*.bin")) + list(staging_dir.rglob("*.zip")):
        if p.is_file() and p.stat().st_size == EXPECTED_ARCHIVE_SIZE:
            archive_candidate = p
            break

assert archive_candidate is not None and archive_candidate.is_file(), (
    f"FAIL-CLOSED: Authorized archive not found under {staging_dir}!\n"
    f"Expected authorized_RQ1-TRN-02A_fold1_train_val_only.bin or .zip (or matching size {EXPECTED_ARCHIVE_SIZE:,} bytes)."
)

# Compute its SHA-256 and assert exact match with b750eb3bf3971afccd9ab9058a64fa6b3f5781399b9ef0f1294d29bf580784a9
print(f"[DATASET] Found archive file: {archive_candidate} (size: {archive_candidate.stat().st_size:,} bytes)")
print("[DATASET] Computing SHA-256 hash for cryptographic verification...")

hasher = hashlib.sha256()
with open(archive_candidate, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        hasher.update(chunk)
computed_sha256 = hasher.hexdigest()

print(f"[DATASET] Archive SHA-256: {computed_sha256}")
assert computed_sha256 == EXPECTED_ARCHIVE_SHA256, (
    f"FAIL-CLOSED: Archive SHA-256 mismatch!\n"
    f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
    f"Actual:   {computed_sha256}\n"
    f"File:     {archive_candidate}\n"
    f"Never fall back to another dataset!"
)
print("[DATASET] Cryptographic checksum PASS. Archive is authentic and verified.")

# 4. Prepare local output directory
output_dir = Path("/kaggle/working/checkpoints/RQ1-P-SUP/b_control")
output_dir.mkdir(parents=True, exist_ok=True)

# 5. Execute Training via audit_scripts/train_p_sup.py
train_cmd = [
    sys.executable,
    "audit_scripts/train_p_sup.py",
    "--arm", "control",
    "--objective", "binary_collapsed",
    "--split-manifest", "manifests/RQ1-SPLIT-01/l2arctic_4fold_splits.json",
    "--zip-path", str(archive_candidate),
    "--output-dir", str(output_dir),
    "--seed", "1337",
    "--batch-size", "2",
    "--gradient-accumulation-steps", "2",
    "--warmup-updates", "100",
    "--max-updates", "3600",
    "--lr", "0.001",
    "--weight-decay", "0.01",
    "--gradient-clip", "1.0",
    "--max-epochs", "25",
    "--train-partition", "fit",
    "--select-partition", "selection",
    "--cal-partition", "calibration",
    "--val-partition", "val",
    "--remote-execution",
    "--remote-mirror-target", "minkhoaa/langfens-rq1-cross-fold-checkpoints",
    "--remote-mirror-backend", "huggingface",
]

print("=" * 80)
print("[LAUNCH] Executing P-SUP Control B-CONTROL Training:")
print("  " + " ".join(train_cmd))
print("=" * 80)
subprocess.run(train_cmd, check=True)
print("[LAUNCH] Control B-CONTROL Training completed successfully.")

In [ ]:
# Phase 4: Post-Training Durability Verification & Remote Mirror Readback
import hashlib, os
from pathlib import Path
from huggingface_hub import HfApi

output_dir = Path("/kaggle/working/checkpoints/RQ1-P-SUP/b_control")
remote_repo = "minkhoaa/langfens-rq1-cross-fold-checkpoints"
remote_prefix = "psup/b_control"

api = HfApi(token=os.environ["HF_TOKEN"])

print("=" * 80)
print(f"[DURABILITY] Verifying local artifacts in {output_dir}:")
for ckpt_name in ("fixed_budget_3600.pt", "fixed_budget_checkpoint.pt", "best.pt", "last.pt", "training_summary.json", "history.json"):
    p = output_dir / ckpt_name
    if not p.is_file():
        for sub in output_dir.rglob(ckpt_name):
            p = sub
            break
    if p.is_file():
        sha = hashlib.sha256(p.read_bytes()).hexdigest()
        print(f"  [LOCAL] {ckpt_name}: {p.stat().st_size:,} bytes | SHA-256: {sha}")
        rel_path = f"{remote_prefix}/{ckpt_name}"
        print(f"  [REMOTE] Ensuring remote mirror at {remote_repo}:{rel_path}...")
        api.upload_file(
            path_or_fileobj=str(p),
            path_in_repo=rel_path,
            repo_id=remote_repo,
            repo_type="model"
        )
        print(f"  -> Confirmed mirrored: {rel_path}")
    else:
        print(f"  [NOTE] Optional/alias artifact {ckpt_name} not present.")

print("[DURABILITY] Durability readback and remote mirroring completed successfully.")
print("=" * 80)


In [ ]:
# Phase 5: Final Execution Summary & Receipt
import json
from pathlib import Path

summary_files = list(Path("/kaggle/working/checkpoints/RQ1-P-SUP/b_control").rglob("training_summary.json"))
if summary_files:
    summary_data = json.loads(summary_files[0].read_text())
    print("=" * 80)
    print("LANGFENS P-SUP B-CONTROL TRAINING SUMMARY RECEIPT:")
    print(json.dumps(summary_data, indent=2))
    print("=" * 80)
else:
    print("[RECEIPT] Execution completed.")
